In [1]:
import pandas as pd

df = pd.read_parquet("../data_lake/gold/prediction_dataset")
print(df.head(10))
print("Total records in prediction dataset: ", df.shape)
print("DataFrame columns: ", df.columns.tolist())

  constructor_id code  season  round              race_name        date  \
0         alpine  ALO    2022      3  Australian Grand Prix  2022-04-10   
1       mercedes  RUS    2022      5       Miami Grand Prix  2022-05-08   
2       williams  ALB    2022      6     Spanish Grand Prix  2022-05-22   
3     alphatauri  TSU    2022      7      Monaco Grand Prix  2022-05-29   
4        mclaren  NOR    2022     13   Hungarian Grand Prix  2022-07-31   
5        mclaren  RIC    2022     17   Singapore Grand Prix  2022-10-02   
6       red_bull  VER    2022      8  Azerbaijan Grand Prix  2022-06-12   
7       williams  LAT    2022     13   Hungarian Grand Prix  2022-07-31   
8       mercedes  RUS    2022     18    Japanese Grand Prix  2022-10-09   
9           haas  MAG    2022     11    Austrian Grand Prix  2022-07-10   

      circuit_id                    circuit_name        driver_id given_name  \
0    albert_park  Albert Park Grand Prix Circuit           alonso   Fernando   
1          mia

In [2]:
# Sort dataset by date to conserve the time series order
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by="date")
print(df.head(5))
print("DataFrame columns: ", df.columns.tolist())

    constructor_id code  season  round           race_name       date  \
479   aston_martin  STR    2021      1  Bahrain Grand Prix 2021-03-28   
835       williams  RUS    2021      1  Bahrain Grand Prix 2021-03-28   
857       red_bull  VER    2021      1  Bahrain Grand Prix 2021-03-28   
490         alpine  ALO    2021      1  Bahrain Grand Prix 2021-03-28   
619     alphatauri  GAS    2021      1  Bahrain Grand Prix 2021-03-28   

    circuit_id                   circuit_name       driver_id given_name  ...  \
479    bahrain  Bahrain International Circuit          stroll      Lance  ...   
835    bahrain  Bahrain International Circuit         russell     George  ...   
857    bahrain  Bahrain International Circuit  max_verstappen        Max  ...   
490    bahrain  Bahrain International Circuit          alonso   Fernando  ...   
619    bahrain  Bahrain International Circuit           gasly     Pierre  ...   

    driver_top10_finishes driver_podium_finishes driver_avg_position_gain 

In [3]:
# Define target variable and feature columns
target = "finished_top_10"

feature_cols = [
    "season",
    "round",
    "grid",
    "circuit_id",
    "driver_id",
    "constructor_id",
    "driver_avg_points",
    "driver_avg_position",
    "driver_num_races",
    "driver_top10_finishes",
    "driver_podium_finishes",
    "driver_avg_position_gain",
    "constructor_avg_points",
    "constructor_avg_position",
    "constructor_num_races",
    "constructor_top10_finishes",
    "constructor_podium_finishes",
    "constructor_avg_position_gain"
]

X = df[feature_cols]
y = df[target]

In [4]:
# Use one-hot encoding for categorical features
X = pd.get_dummies(X, columns=["circuit_id", "driver_id", "constructor_id"], drop_first=True)
X.head(5)

,season,round,grid,driver_avg_points,driver_avg_position,driver_num_races,driver_top10_finishes,driver_podium_finishes,driver_avg_position_gain,constructor_avg_points,...,driver_id_zhou,constructor_id_alphatauri,constructor_id_alpine,constructor_id_aston_martin,constructor_id_ferrari,constructor_id_haas,constructor_id_mclaren,constructor_id_mercedes,constructor_id_red_bull,constructor_id_williams
479,2021,1,10,1.818182,11.727273,66,29,0,1.045455,3.015152,...,False,False,False,True,False,False,False,False,False,False
835,2021,1,15,6.590909,9.000000,66,41,11,-0.696970,0.431818,...,False,False,False,False,False,False,False,False,False,True
857,2021,1,1,20.477273,2.969697,66,61,56,0.181818,15.852273,...,False,False,False,False,False,False,False,False,True,False
490,2021,1,9,5.454545,9.136364,66,48,9,-0.196970,3.295455,...,False,False,True,False,False,False,False,False,False,False
619,2021,1,5,2.833333,10.833333,66,32,2,-1.409091,1.507576,...,False,True,False,False,False,False,False,False,False,False


In [16]:
# Split the data into training and testing sets
split_index = int(len(df) * 0.8)

X_train = X[:split_index]
y_train = y[:split_index]

X_test = X[split_index:]
y_test = y[split_index:]

In [17]:
import mlflow
import mlflow.sklearn

# Set up MLflow tracking
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Default")
print(mlflow.get_tracking_uri())

# Logistic Regression Model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

with mlflow.start_run(run_name="Logistic Regression"):
    log_model = LogisticRegression(max_iter=1000)

    log_model.fit(X_train, y_train)

    y_pred_log = log_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred_log)
    report = classification_report(y_test, y_pred_log)
    cm = confusion_matrix(y_test, y_pred_log)

    mlflow.sklearn.log_model(log_model, "logistic_regression_model")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_text(report, "classification_report.txt")
    mlflow.log_text(str(cm), "confusion_matrix.txt")

MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Default (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] Es konnte keine Verbindung hergestellt werden, da der Zielcomputer die Verbindung verweigerte"))

In [ ]:
#Random Forest Model
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="Random Forest"):
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

    rf_model.fit(X_train, y_train)

    y_pred_rf = rf_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred_rf)
    report = classification_report(y_test, y_pred_rf)
    cm = confusion_matrix(y_test, y_pred_rf)

    mlflow.sklearn.log_model(rf_model, "random_forest_model")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_text(report, "classification_report.txt")
    mlflow.log_text(str(cm), "confusion_matrix.txt")

In [ ]:
# Save the best model (Random Forest) to a file to be used in the API
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)
joblib.dump(rf_model, "../models/best_model.pkl")